# Tiny LLM — Revised DPO Training & Evaluation

**Improvements over the original DPO notebook:**
- Blended dataset: `orca_dpo` + `hh_rlhf` for diverse, human-grounded preference signal
- Length-normalized log-probs (removes bias toward shorter responses)
- Fixed EOS truncation bug (EOS token is always preserved)
- Consistent tokenizer (`GPT2TokenizerFast`, matching pretraining & SFT)
- Hyperparameter sweep: beta in {0.1, 0.3, 0.5} x epochs in {1, 3} x 2 SFT checkpoints
- Extended evaluation: advantages distribution, response length analysis, qualitative comparison

```
Pretrained checkpoint --> SFT --> DPO
                                 ^^^^ this notebook
```

In [ ]:
# Re-install in editable mode so the kernel picks up all source changes
# (new config fields, tokenizer fix, EOS fix, length normalization).
# After running this cell, RESTART THE KERNEL before continuing.
!pip install -e '../[all]' -q

## 0. Setup

In [ ]:
import os, pathlib, sys, subprocess, json, time, glob
import numpy as np

_cwd = pathlib.Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / 'pyproject.toml').exists() and (p / 'src').exists()),
    _cwd,
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))

print(f'Working directory: {REPO_ROOT}')

# ---------- Configuration ----------
DPO_CONFIG     = 'configs/dpo.toml'
DEVICE         = 'auto'     # 'cuda', 'mps', 'cpu', or 'auto'
TRAINING_MODE  = 'smoke'    # 'smoke' | 'quick' | 'full'
SWEEP_DIR      = pathlib.Path('checkpoints/dpo_sweep')

# Samples per mode (controls dataset size and thus training time)
MODE_SAMPLES = {'smoke': 200, 'quick': 2000, 'full': None}
print(f'Mode: {TRAINING_MODE} (max_samples={MODE_SAMPLES[TRAINING_MODE]})')

# MPS memory safety: reduce batch size on Apple Silicon (24 GB unified)
# DPO needs 4 forward passes per micro-batch, so batch_size=2 + higher
# grad_accum keeps the effective batch at 32 while fitting in memory.
import torch
_device = torch.device('mps' if torch.backends.mps.is_available() else
                       'cuda' if torch.cuda.is_available() else 'cpu')
if _device.type == 'mps':
    MPS_OVERRIDES = [
        'posttraining.batch_size=2',
        'posttraining.grad_accum_steps=16',
    ]
    print(f'MPS detected: will use batch_size=2, grad_accum=16 (effective=32)')
else:
    MPS_OVERRIDES = []

### Verify SFT Checkpoints

In [ ]:
import torch

sft_dir = pathlib.Path('checkpoints/sft')
SFT_CHECKPOINTS = sorted(sft_dir.glob('*.pt'))

if not SFT_CHECKPOINTS:
    raise FileNotFoundError(f'No SFT checkpoints in {sft_dir}')

for i, p in enumerate(SFT_CHECKPOINTS):
    meta = torch.load(str(p), map_location='cpu', weights_only=False)
    mc = meta.get('model_config', {})
    n_params = sum(v.numel() for v in meta.get('model_state_dict', meta.get('model', {})).values())
    print(f'[{i}] {p.name}')
    print(f'    Size: {p.stat().st_size / 1e6:.1f} MB | Params: {n_params:,}')
    print(f'    chat_template: {mc.get("chat_template", "not set")}')
    del meta

print(f'\n{len(SFT_CHECKPOINTS)} SFT checkpoint(s) ready for DPO.')

## 1. Blended Dataset: Preparation & Exploration

We blend two preference datasets:
- **Orca DPO** (`argilla/distilabel-intel-orca-dpo-pairs`): GPT-4 vs GPT-3.5 instruction-following pairs
- **HH-RLHF** (`Anthropic/hh-rlhf`): Human-annotated helpfulness + harmlessness preferences

The blend gives our 40M model both instruction-quality signal (Orca) and human-grounded preference signal (HH-RLHF).

In [ ]:
from tiny_llm.posttraining.dpo_dataset import prepare_dpo_dataset, _get_tokenizer
from tiny_llm.utils.config import load_config

max_samples = MODE_SAMPLES[TRAINING_MODE]
cfg = load_config(DPO_CONFIG)
post_cfg = cfg.posttraining.model_dump()

dpo_datasets = post_cfg.get('dpo_datasets', ['orca_dpo', 'hh_rlhf'])
length_norm  = post_cfg.get('dpo_length_normalized', True)

print(f'Datasets: {dpo_datasets}')
print(f'Length normalization: {length_norm}')
print(f'Max seq len: {post_cfg["max_seq_len"]}')

train_examples, val_examples = prepare_dpo_dataset(
    datasets=dpo_datasets,
    cache_dir=post_cfg.get('dpo_dataset_dir', 'data/dpo'),
    max_seq_len=int(post_cfg['max_seq_len']),
    val_fraction=float(post_cfg['val_fraction']),
    max_samples=max_samples,
    seed=42,
    force=True,  # rebuild cache with fixed tokenizer & EOS
)

print(f'\nBlended dataset: {len(train_examples):,} train + {len(val_examples):,} val')

In [ ]:
import matplotlib.pyplot as plt

enc = _get_tokenizer()

prompt_lens  = [len(ex.prompt_ids) for ex in train_examples]
chosen_lens  = [len(ex.chosen_ids) for ex in train_examples]
rejected_lens = [len(ex.rejected_ids) for ex in train_examples]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Token Length Distributions (n={len(train_examples):,})',
             fontsize=13, fontweight='bold')

axes[0].hist(prompt_lens, bins=50, alpha=0.7, color='#1f77b4')
axes[0].set_title('Prompt Length')
axes[0].set_xlabel('Tokens')
axes[0].axvline(np.median(prompt_lens), color='red', ls='--',
                label=f'median={np.median(prompt_lens):.0f}')
axes[0].legend()

axes[1].hist(chosen_lens, bins=50, alpha=0.7, color='#2ca02c', label='Chosen')
axes[1].hist(rejected_lens, bins=50, alpha=0.5, color='#d62728', label='Rejected')
axes[1].set_title('Response Lengths')
axes[1].set_xlabel('Tokens')
axes[1].legend()

len_diffs = [c - r for c, r in zip(chosen_lens, rejected_lens)]
axes[2].hist(len_diffs, bins=50, alpha=0.7, color='#9467bd')
axes[2].axvline(0, color='black', ls='-', lw=0.8)
axes[2].set_title('Chosen - Rejected Length')
axes[2].set_xlabel('Token difference')

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f'Prompt:   mean={np.mean(prompt_lens):.0f}, median={np.median(prompt_lens):.0f}')
print(f'Chosen:   mean={np.mean(chosen_lens):.0f}, median={np.median(chosen_lens):.0f}')
print(f'Rejected: mean={np.mean(rejected_lens):.0f}, median={np.median(rejected_lens):.0f}')
print(f'Len diff: mean={np.mean(len_diffs):.1f} (positive = chosen is longer)')

# Verify EOS fix
eos_ok = all(ex.chosen_ids[-1] == 50256 and ex.rejected_ids[-1] == 50256
             for ex in train_examples)
print(f'\nEOS token always present: {eos_ok}')

In [ ]:
# Show example pairs
for i, ex in enumerate(train_examples[:3]):
    print(f'=== Example {i} ===')
    print(f'Prompt  ({len(ex.prompt_ids):>4} tok): {enc.decode(ex.prompt_ids)[:200]}...')
    print(f'Chosen  ({len(ex.chosen_ids):>4} tok): {enc.decode(ex.chosen_ids)[:150]}...')
    print(f'Rejected({len(ex.rejected_ids):>4} tok): {enc.decode(ex.rejected_ids)[:150]}...')
    print()

## 2. Experiment Grid

We sweep over:
- **2 SFT checkpoints** (different LR / dataset configs)
- **3 beta values**: 0.1 (original DPO default), 0.3, 0.5 (stronger KL penalty)
- **2 epoch counts**: 1 (quick, risk underfitting) vs 3 (more signal, risk overfitting)

All runs use **length-normalized** log-probs and the **blended** dataset.

In [ ]:
BETAS = [0.1, 0.3, 0.5]
EPOCH_COUNTS = [1, 3]

experiments = []
for ckpt_idx, ckpt_path in enumerate(SFT_CHECKPOINTS):
    ckpt_label = f'sft{ckpt_idx + 1}'
    for beta in BETAS:
        for epochs in EPOCH_COUNTS:
            name = f'{ckpt_label}_b{beta}_e{epochs}'
            experiments.append({
                'name': name,
                'checkpoint': str(ckpt_path),
                'ckpt_label': ckpt_label,
                'beta': beta,
                'epochs': epochs,
                'output_dir': str(SWEEP_DIR / name),
            })

print(f'Total experiments: {len(experiments)}')
print(f'{"":<3} {"Name":<25} {"SFT Checkpoint":<45} {"beta":>5} {"Epochs":>6}')
print('-' * 88)
for i, exp in enumerate(experiments):
    print(f'{i:<3} {exp["name"]:<25} {pathlib.Path(exp["checkpoint"]).name:<45} '
          f'{exp["beta"]:>5} {exp["epochs"]:>6}')

## 3. Run Experiments

Each experiment runs the DPO training pipeline via CLI with the appropriate overrides.
Already-completed experiments (those with existing checkpoints) are skipped.

> **MPS users:** If you hit OOM, reduce `batch_size` or `max_seq_len` via overrides.
> On CUDA with >=16 GB VRAM, the default settings should work fine.

In [ ]:
# Per-mode training overrides
MODE_OVERRIDES = {
    'smoke': [
        'posttraining.max_samples=200',
        'posttraining.eval_interval=5',
        'posttraining.checkpoint_interval=999',
        'posttraining.log_interval=1',
        'posttraining.pretrain_eval_interval=999',
    ],
    'quick': [
        'posttraining.max_samples=2000',
        'posttraining.eval_interval=25',
    ],
    'full': [],
}


def run_experiment(exp, force=False):
    """Run one DPO experiment. Returns True if successful."""
    out_dir = pathlib.Path(exp['output_dir'])

    # Skip if already completed
    existing = sorted(out_dir.glob('step_*.pt'))
    if existing and not force:
        print(f'  [SKIP] {exp["name"]}: checkpoint(s) exist. Set force=True to rerun.')
        return True

    overrides = [
        f'posttraining.dpo_beta={exp["beta"]}',
        f'posttraining.epochs={exp["epochs"]}',
        f'posttraining.checkpoint_dir={exp["output_dir"]}',
        'posttraining.dpo_length_normalized=true',
    ] + MODE_OVERRIDES.get(TRAINING_MODE, []) + MPS_OVERRIDES

    cmd = [
        'uv', 'run', 'python', 'main.py',
        '--stage', 'posttraining',
        '--config', DPO_CONFIG,
        '--checkpoint', exp['checkpoint'],
        '--device', DEVICE,
        '--overrides',
    ] + overrides

    # Disable pretrain-val probe if data not available
    if not pathlib.Path('data/tokenized/val.bin').exists():
        cmd.append('posttraining.pretrain_val_path=')

    print(f'  [RUN]  {exp["name"]}: beta={exp["beta"]}, epochs={exp["epochs"]}')

    t0 = time.time()
    proc = subprocess.run(
        cmd, cwd=str(REPO_ROOT),
        capture_output=True, text=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )
    elapsed = time.time() - t0

    if proc.returncode != 0:
        print(f'  [FAIL] {exp["name"]} ({elapsed:.0f}s)')
        lines = (proc.stdout + proc.stderr).strip().split('\n')
        for line in lines[-15:]:
            print(f'    {line}')
        return False

    print(f'  [OK]   {exp["name"]} ({elapsed:.0f}s)')
    return True


# --- Run all experiments ---
print(f'Running {len(experiments)} experiments (mode={TRAINING_MODE})\n')
run_results = {}
for exp in experiments:
    ok = run_experiment(exp)
    run_results[exp['name']] = ok

succeeded = sum(run_results.values())
print(f'\n{"="*60}')
print(f'Completed: {succeeded}/{len(experiments)} experiments')
if succeeded < len(experiments):
    failed = [k for k, v in run_results.items() if not v]
    print(f'Failed: {failed}')

## 4. Comparative Results Dashboard

Load metrics from all completed experiments and visualize side by side.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def load_metrics(exp):
    """Load metrics.csv for one experiment, return None if not found."""
    csv_path = pathlib.Path(exp['output_dir']) / 'metrics.csv'
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path)
    df['experiment'] = exp['name']
    df['beta'] = exp['beta']
    df['epochs'] = exp['epochs']
    df['ckpt_label'] = exp['ckpt_label']
    return df


all_metrics = []
for exp in experiments:
    df = load_metrics(exp)
    if df is not None:
        all_metrics.append(df)

if not all_metrics:
    print('No metrics found -- run experiments first (Section 3).')
else:
    combined = pd.concat(all_metrics, ignore_index=True)
    exp_names = combined['experiment'].unique()
    n_exps = len(exp_names)
    colors = plt.cm.tab10(np.linspace(0, 1, max(n_exps, 1)))

    # Only plot columns that actually exist in the data
    candidate_panels = [
        ('train/loss',                    'Train Loss',              'Loss'),
        ('train/accuracy',                'Train Accuracy',          'Accuracy'),
        ('val/loss',                      'Val Loss',                'Loss'),
        ('train/implicit_reward_margin',  'Implicit Reward Margin',  'Margin'),
    ]
    panels = [(col, t, y) for col, t, y in candidate_panels if col in combined.columns]

    n_panels = len(panels)
    ncols = min(n_panels, 2)
    nrows = (n_panels + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 5 * nrows), squeeze=False)
    fig.suptitle('DPO Sweep -- Comparative Dashboard', fontsize=14, fontweight='bold')

    for pidx, (col, title, ylabel) in enumerate(panels):
        ax = axes[pidx // ncols, pidx % ncols]
        for i, name in enumerate(exp_names):
            sub = combined[combined['experiment'] == name]
            mask = sub[col].notna()
            if mask.any():
                style = dict(color=colors[i], linewidth=1.0, label=name, alpha=0.8)
                if 'val' in col:
                    style.update(linewidth=1.2, marker='o', markersize=3)
                ax.plot(sub.loc[mask, 'step'], sub.loc[mask, col], **style)
        ax.set_title(title)
        ax.set_xlabel('Step')
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7, loc='best')
        if 'accuracy' in col.lower():
            ax.set_ylim(-0.05, 1.05)

    # Hide unused subplot slots
    for pidx in range(n_panels, nrows * ncols):
        axes[pidx // ncols, pidx % ncols].set_visible(False)

    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()
    print(f'Loaded metrics from {n_exps} experiments.')

In [ ]:
# Summary table
if all_metrics:
    summary_rows = []
    for exp in experiments:
        df = load_metrics(exp)
        if df is None:
            continue
        train_rows = df.dropna(subset=['train/loss'])

        row = {
            'Experiment': exp['name'],
            'SFT': exp['ckpt_label'],
            'Beta': exp['beta'],
            'Epochs': exp['epochs'],
            'Steps': int(train_rows['step'].max()) if not train_rows.empty else 0,
            'Final Loss': round(train_rows.iloc[-1]['train/loss'], 4) if not train_rows.empty else None,
        }

        # These columns may not exist in older metrics CSVs
        if 'train/accuracy' in df.columns:
            acc_rows = df.dropna(subset=['train/accuracy'])
            row['Final Acc'] = round(acc_rows.iloc[-1]['train/accuracy'], 3) if not acc_rows.empty else None
        if 'val/loss' in df.columns:
            val_rows = df.dropna(subset=['val/loss'])
            row['Best Val Loss'] = round(val_rows['val/loss'].min(), 4) if not val_rows.empty else None
        if 'val/accuracy' in df.columns:
            va_rows = df.dropna(subset=['val/accuracy'])
            row['Best Val Acc'] = round(va_rows['val/accuracy'].max(), 3) if not va_rows.empty else None

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)

    # Highlight best config per SFT checkpoint
    for sft in summary_df['SFT'].unique():
        subset = summary_df[summary_df['SFT'] == sft]
        if 'Best Val Loss' in subset.columns and subset['Best Val Loss'].notna().any():
            best_idx = subset['Best Val Loss'].idxmin()
            print(f'\nBest for {sft}: {subset.loc[best_idx, "Experiment"]} '
                  f'(val_loss={subset.loc[best_idx, "Best Val Loss"]})')
        elif 'Final Acc' in subset.columns and subset['Final Acc'].notna().any():
            best_idx = subset['Final Acc'].idxmax()
            print(f'\nBest for {sft} (by train acc): {subset.loc[best_idx, "Experiment"]} '
                  f'(acc={subset.loc[best_idx, "Final Acc"]})')

## 5. Extended Evaluation

Beyond loss and accuracy, we examine:
1. **Advantages distribution** -- if advantages collapse to extreme values, the model has diverged
2. **Response length analysis** -- detect length bias (are DPO models generating shorter/longer?)
3. **Qualitative comparison** -- side-by-side generations on fixed prompts

### 5a. Advantages Distribution

Healthy DPO training shows advantages gradually moving positive (chosen preferred).
If advantages explode or collapse, the beta is too low or training has diverged.

In [ ]:
if all_metrics:
    completed = [exp for exp in experiments if load_metrics(exp) is not None]
    # Check if any experiment has the advantages column
    has_adv = any('train/advantages' in load_metrics(e).columns for e in completed)

    if not has_adv:
        print('train/advantages column not found in metrics CSVs.')
        print('This column is logged by the updated trainer. Re-run experiments')
        print('after pushing the latest source code to get full metrics.')
    else:
        adv_exps = [e for e in completed if 'train/advantages' in load_metrics(e).columns]
        n = len(adv_exps)
        cols = min(3, n) if n > 0 else 1
        rows = max(1, (n + cols - 1) // cols)

        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
        fig.suptitle('Advantages Distribution per Experiment', fontsize=13, fontweight='bold')

        for idx, exp in enumerate(adv_exps):
            ax = axes[idx // cols, idx % cols]
            df = load_metrics(exp)
            adv = df['train/advantages'].dropna()
            if not adv.empty:
                ax.hist(adv, bins=30, alpha=0.7, color='#2ca02c', edgecolor='white')
                ax.axvline(0, color='red', ls='--', lw=0.8)
                ax.axvline(adv.mean(), color='blue', ls='--', lw=0.8,
                           label=f'mean={adv.mean():.3f}')
                ax.axvline(adv.iloc[-1], color='orange', ls=':', lw=0.8,
                           label=f'final={adv.iloc[-1]:.3f}')
                ax.set_title(exp['name'], fontsize=10)
                ax.set_xlabel('Advantages')
                ax.legend(fontsize=7)
                ax.grid(True, alpha=0.3)

        for idx in range(n, rows * cols):
            axes[idx // cols, idx % cols].set_visible(False)

        fig.tight_layout(rect=(0, 0, 1, 0.94))
        plt.show()

        # Print advantages summary
        print(f'{"Experiment":<25} {"Mean Adv":>10} {"Std Adv":>10} {"Final Adv":>10}')
        print('-' * 60)
        for exp in adv_exps:
            df = load_metrics(exp)
            adv = df['train/advantages'].dropna()
            if not adv.empty:
                print(f'{exp["name"]:<25} {adv.mean():>10.4f} {adv.std():>10.4f} {adv.iloc[-1]:>10.4f}')

### 5b. Response Length & Qualitative Comparison

Generate responses from each model on a fixed set of prompts.
This lets us check:
- Whether DPO models produce shorter/longer responses than SFT (length bias)
- Qualitative differences in response style and content

In [ ]:
EVAL_PROMPTS = [
    "Explain what gradient descent is in simple terms.",
    "What is the capital of France?",
    "Write a haiku about machine learning.",
    "Summarize the key ideas behind transformers.",
    "Why is the sky blue?",
]

# Detect whether to use 'uv run python' or just 'python'
import shutil
_PYTHON_CMD = ['uv', 'run', 'python'] if shutil.which('uv') else ['python']
print(f'Using: {" ".join(_PYTHON_CMD)}')

# Resolve device once (avoid passing 'auto' to subprocess -- older code may not handle it)
_RESOLVED_DEVICE = _device.type  # from setup cell: 'cuda', 'mps', or 'cpu'
print(f'Inference device: {_RESOLVED_DEVICE}')


def generate_response(checkpoint, prompt, max_tokens=120, temperature=0.7):
    """Generate text from a checkpoint using the CLI."""
    try:
        r = subprocess.run(
            _PYTHON_CMD + [
                'main.py', '--stage', 'inference',
                '--checkpoint', checkpoint, '--chat-template', 'alpaca',
                '--prompt', prompt, '--max-tokens', str(max_tokens),
                '--temperature', str(temperature), '--device', _RESOLVED_DEVICE,
                '--leaderboard',
            ],
            cwd=str(REPO_ROOT), capture_output=True, text=True, timeout=120,
        )
        if r.returncode != 0:
            err = (r.stderr or r.stdout or '').strip().split('\n')
            err_msg = next((l for l in reversed(err) if l.strip()), 'unknown error')
            return f'[ERROR: {err_msg[:120]}]'
        output = r.stdout.strip()
        return output if output else '[EMPTY]'
    except subprocess.TimeoutExpired:
        return '[TIMEOUT]'


def find_best_checkpoint(exp_dir):
    """Find best or latest checkpoint in an experiment directory."""
    d = pathlib.Path(exp_dir)
    best = sorted(d.glob('best_*.pt'))
    if best:
        return str(best[-1])
    steps = sorted(d.glob('step_*.pt'))
    if steps:
        return str(steps[-1])
    return None

In [ ]:
# Collect all checkpoints to evaluate
checkpoints_to_eval = {}

# SFT baselines
for i, ckpt in enumerate(SFT_CHECKPOINTS):
    checkpoints_to_eval[f'SFT-{i+1}'] = str(ckpt)

# DPO experiments
for exp in experiments:
    ckpt = find_best_checkpoint(exp['output_dir'])
    if ckpt:
        checkpoints_to_eval[exp['name']] = ckpt

print(f'Generating from {len(checkpoints_to_eval)} models x '
      f'{len(EVAL_PROMPTS)} prompts ...\n')

gen_results = {}
for label, ckpt in checkpoints_to_eval.items():
    gen_results[label] = {}
    for prompt in EVAL_PROMPTS:
        gen_results[label][prompt] = generate_response(ckpt, prompt)
    print(f'  [OK] {label}')

print(f'\nDone. Generated {sum(len(v) for v in gen_results.values())} responses.')

In [ ]:
# --- Response Length Analysis ---
enc = _get_tokenizer()

# Adapt to whichever tokenizer is loaded (tiktoken vs GPT2TokenizerFast)
def _tokenize(text):
    try:
        return enc.encode(text, add_special_tokens=False)
    except TypeError:
        return enc.encode(text, disallowed_special=())

print('=' * 80)
print('RESPONSE LENGTH ANALYSIS (tokens)')
print('=' * 80)

length_data = {}
for label in gen_results:
    lengths = []
    for prompt in EVAL_PROMPTS:
        text = gen_results[label][prompt]
        lengths.append(len(_tokenize(text)))
    length_data[label] = lengths
    print(f'  {label:<30} mean={np.mean(lengths):5.1f}  '
          f'std={np.std(lengths):5.1f}  '
          f'min={min(lengths):3d}  max={max(lengths):3d}')

# Bar chart
if length_data:
    fig, ax = plt.subplots(figsize=(14, 5))
    labels_list = list(length_data.keys())
    x = np.arange(len(EVAL_PROMPTS))
    width = 0.8 / max(len(labels_list), 1)

    for i, label in enumerate(labels_list):
        ax.bar(x + i * width, length_data[label], width, label=label, alpha=0.8)

    ax.set_xlabel('Prompt')
    ax.set_ylabel('Response Length (tokens)')
    ax.set_title('Response Length per Model & Prompt')
    ax.set_xticks(x + width * len(labels_list) / 2)
    short_prompts = [p[:35] + '...' if len(p) > 35 else p for p in EVAL_PROMPTS]
    ax.set_xticklabels(short_prompts, rotation=15, ha='right', fontsize=8)
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3, axis='y')
    fig.tight_layout()
    plt.show()

# Mean response length comparison
if length_data:
    print('\nMean response length (lower may indicate length bias):')
    for label in sorted(length_data, key=lambda k: np.mean(length_data[k])):
        m = np.mean(length_data[label])
        is_sft = label.startswith('SFT')
        marker = ' (baseline)' if is_sft else ''
        print(f'  {label:<30} {m:5.1f} tokens{marker}')

In [ ]:
# --- Side-by-Side Qualitative Comparison ---
print('=' * 80)
print('QUALITATIVE COMPARISON -- Side by Side')
print('=' * 80)

for prompt in EVAL_PROMPTS:
    print(f'\n{"_"*80}')
    print(f'PROMPT: {prompt}')
    print(f'{"_"*80}')
    for label in gen_results:
        response = gen_results[label][prompt]
        if len(response) > 300:
            response = response[:300] + '...'
        print(f'\n  [{label}]:')
        print(f'    {response}')

## 6. Best Config Selection & Final Comparison

Pick the best DPO config (by val loss or accuracy) and compare head-to-head with the SFT baseline.

In [ ]:
# Identify best DPO experiment per SFT checkpoint
best_dpo = {}

for sft_label in sorted(set(e['ckpt_label'] for e in experiments)):
    sft_exps = [e for e in experiments if e['ckpt_label'] == sft_label]
    best_score = float('inf')
    best_exp = None

    for exp in sft_exps:
        df = load_metrics(exp)
        if df is None:
            continue
        # Prefer val loss if available, else use final train loss
        if 'val/loss' in df.columns and df['val/loss'].notna().any():
            score = df['val/loss'].min()
        elif 'train/loss' in df.columns and df['train/loss'].notna().any():
            score = df['train/loss'].iloc[-1]
        else:
            continue

        if score < best_score:
            best_score = score
            best_exp = exp

    if best_exp:
        best_dpo[sft_label] = best_exp
        print(f'{sft_label}: best DPO = {best_exp["name"]} (score={best_score:.4f})')

if not best_dpo:
    print('No completed experiments found.')

In [ ]:
# Head-to-head: SFT baseline vs best DPO for each checkpoint
if best_dpo and gen_results:
    for sft_label, dpo_exp in best_dpo.items():
        sft_key = f'SFT-{sft_label[-1]}'  # sft1 -> SFT-1
        dpo_key = dpo_exp['name']

        if sft_key not in gen_results or dpo_key not in gen_results:
            continue

        print(f'\n{"="*80}')
        print(f'{sft_label.upper()}: {sft_key} vs {dpo_key}')
        print(f'{"="*80}')

        for prompt in EVAL_PROMPTS:
            sft_resp = gen_results[sft_key][prompt]
            dpo_resp = gen_results[dpo_key][prompt]

            if len(sft_resp) > 250:
                sft_resp = sft_resp[:250] + '...'
            if len(dpo_resp) > 250:
                dpo_resp = dpo_resp[:250] + '...'

            print(f'\n  Prompt: {prompt}')
            print(f'  SFT:  {sft_resp}')
            print(f'  DPO:  {dpo_resp}')

In [ ]:
# --- Beta sensitivity plot ---
# Show how beta affects final metrics, grouped by SFT checkpoint and epoch count
if all_metrics:
    metric_cols = [
        ('train/loss', 'Final Train Loss'),
        ('train/accuracy', 'Final Train Accuracy'),
        ('train/implicit_reward_margin', 'Final Reward Margin'),
    ]
    # Only plot columns that exist
    metric_cols = [(col, t) for col, t in metric_cols if col in combined.columns]

    if metric_cols:
        fig, axes = plt.subplots(1, len(metric_cols), figsize=(5 * len(metric_cols), 4))
        if len(metric_cols) == 1:
            axes = [axes]
        fig.suptitle('Beta Sensitivity Analysis', fontsize=13, fontweight='bold')

        markers = {'1': 'o', '3': 's'}
        colors_sft = {'sft1': '#1f77b4', 'sft2': '#ff7f0e'}

        for ax, (col, title) in zip(axes, metric_cols):
            for exp in experiments:
                df = load_metrics(exp)
                if df is None or col not in df.columns:
                    continue
                vals = df[col].dropna()
                if vals.empty:
                    continue
                y = vals.iloc[-1]
                c = colors_sft.get(exp['ckpt_label'], 'gray')
                m = markers.get(str(exp['epochs']), 'o')
                label = f'{exp["ckpt_label"]}, {exp["epochs"]}ep'
                ax.scatter(exp['beta'], y, color=c, marker=m, s=80,
                           label=label, zorder=3)

            ax.set_title(title)
            ax.set_xlabel('Beta')
            ax.set_xticks(BETAS)
            ax.grid(True, alpha=0.3)

        # Deduplicate legend
        handles, labels = axes[0].get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        axes[0].legend(by_label.values(), by_label.keys(), fontsize=8)

        fig.tight_layout(rect=(0, 0, 1, 0.93))
        plt.show()
    else:
        print('No plottable metrics found.')

---

## CLI Quick Reference

```bash
# Single DPO run with the revised config (blended dataset, length norm)
uv run python main.py --stage posttraining --config configs/dpo.toml \
    --checkpoint checkpoints/sft/lr_2e-05_oasst_3_epoch_3_best.pt \
    --device cuda

# Override beta and epochs
uv run python main.py --stage posttraining --config configs/dpo.toml \
    --checkpoint checkpoints/sft/lr_2e-05_oasst_3_epoch_3_best.pt \
    --device cuda \
    --overrides posttraining.dpo_beta=0.3 posttraining.epochs=3

# Smoke test (small subset)
uv run python main.py --stage posttraining --config configs/dpo.toml \
    --checkpoint checkpoints/sft/lr_2e-05_oasst_3_epoch_3_best.pt \
    --overrides posttraining.max_samples=200 posttraining.epochs=1
```

### Key changes from original setup
| Change | Before | After |
|--------|--------|-------|
| Dataset | `orca_dpo` only | `orca_dpo` + `hh_rlhf` blend |
| Log-probs | Summed (length-biased) | Length-normalized |
| EOS token | Could be truncated | Always preserved |
| Tokenizer | `tiktoken` (direct) | `GPT2TokenizerFast` (matches pretraining) |
| Config | Stale SFT fields | Cleaned up |